# Module A: Lightweight DBMS with B+ Tree Index
## CS432 — Databases | Assignment 2

## 1. Introduction

Traditional databases rely on efficient indexing structures to handle large volumes of data.
This report documents the implementation of a lightweight Database Management System (DBMS)
built around a **B+ Tree** indexing engine.

### Problem Statement
Linear search through unsorted data is O(n) — slow for large datasets.
We need a structure that supports fast insert, delete, search, and range queries.

### Solution
A **B+ Tree** provides O(log n) operations with efficient range scans via leaf-level linked lists.
We compare it against a **BruteForceDB** (linear list) to demonstrate the performance gains.

### File Structure
- `bplustree.py` — Core B+ Tree implementation
- `bruteforce.py` — Baseline linear search implementation  
- `table.py` — Table abstraction using B+ Tree
- `db_manager.py` — Database manager for multiple tables

In [1]:
# Imports
import sys
import os
import time
import random
import tracemalloc
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sys.path.append(os.path.abspath('..'))

from Database.bplustree import BPlusTree
from Database.bruteforce import BruteForceDB
from Database.table import Table
from Database.db_manager import DatabaseManager

## 2. Implementation Details

### B+ Tree Properties
- All **data lives in leaf nodes** only
- Internal nodes hold **separator keys** for routing
- Leaf nodes are **linked** via `next` pointers for range scans
- Nodes **split** when full on insert
- Nodes **merge or redistribute** on delete to prevent underflow

### Key Operations
| Operation    | Time Complexity |
|-------------|----------------|
| Search      | O(log n)        |
| Insert      | O(log n)        |
| Delete      | O(log n)        |
| Range Query | O(log n + k)    |
| Get All     | O(n)            |

### BruteForceDB
A simple Python list — all operations are O(n) linear scans.
Used as a baseline to show how much faster the B+ Tree is.

In [2]:
## 3. Basic Operations Demo

db = DatabaseManager("StayEaseDB")

# Create a students table
db.create_table("students", ["student_id", "name", "age", "grade"], order=4)
students = db.get_table("students")

# Insert records
records = [
    {"student_id": 101, "name": "Alice",   "age": 20, "grade": "A"},
    {"student_id": 205, "name": "Bob",     "age": 21, "grade": "B"},
    {"student_id": 150, "name": "Charlie", "age": 19, "grade": "A"},
    {"student_id": 310, "name": "Diana",   "age": 22, "grade": "C"},
    {"student_id": 275, "name": "Eve",     "age": 20, "grade": "B"},
    {"student_id": 189, "name": "Frank",   "age": 23, "grade": "A"},
    {"student_id": 420, "name": "Grace",   "age": 21, "grade": "B"},
]

for r in records:
    students.insert(r)

print("=== All Records (sorted by student_id) ===")
for key, val in students.get_all():
    print(f"  {key}: {val}")

print("\n=== Search: student_id = 205 ===")
print(students.search(205))

print("\n=== Range Query: 150 to 310 ===")
for key, val in students.range_query(150, 310):
    print(f"  {key}: {val}")

print("\n=== Update: student_id = 101 grade -> A+ ===")
students.update(101, {"student_id": 101, "name": "Alice", "age": 20, "grade": "A+"})
print(students.search(101))

print("\n=== Delete: student_id = 205 ===")
students.delete(205)
print("All records after delete:")
for key, val in students.get_all():
    print(f"  {key}: {val}")

Table 'students' created successfully.
=== All Records (sorted by student_id) ===
  101: {'student_id': 101, 'name': 'Alice', 'age': 20, 'grade': 'A'}
  150: {'student_id': 150, 'name': 'Charlie', 'age': 19, 'grade': 'A'}
  189: {'student_id': 189, 'name': 'Frank', 'age': 23, 'grade': 'A'}
  205: {'student_id': 205, 'name': 'Bob', 'age': 21, 'grade': 'B'}
  275: {'student_id': 275, 'name': 'Eve', 'age': 20, 'grade': 'B'}
  310: {'student_id': 310, 'name': 'Diana', 'age': 22, 'grade': 'C'}
  420: {'student_id': 420, 'name': 'Grace', 'age': 21, 'grade': 'B'}

=== Search: student_id = 205 ===
{'student_id': 205, 'name': 'Bob', 'age': 21, 'grade': 'B'}

=== Range Query: 150 to 310 ===
  150: {'student_id': 150, 'name': 'Charlie', 'age': 19, 'grade': 'A'}
  189: {'student_id': 189, 'name': 'Frank', 'age': 23, 'grade': 'A'}
  205: {'student_id': 205, 'name': 'Bob', 'age': 21, 'grade': 'B'}
  275: {'student_id': 275, 'name': 'Eve', 'age': 20, 'grade': 'B'}
  310: {'student_id': 310, 'name': '

In [3]:
## 4. Tree Visualisation

# Build a small tree for clear visualization
viz_tree = BPlusTree(order=4)
for key in [10, 20, 5, 6, 12, 30, 7, 17, 3, 25, 15, 28]:
    viz_tree.insert(key, f"val_{key}")

dot = viz_tree.visualize_tree()
dot.render("bplustree_viz", format="png", cleanup=True)

from IPython.display import Image
Image("bplustree_viz.png")

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH